# RAG-04 · Grounded Answer and Citation Evaluation
### Section 06 — Separate retrieval success from answer success

**Prerequisite:** RAG-03. **Estimated time:** 6–8 hours. The first answerer is
deliberately extractive: it returns cited evidence rather than free-form LLM text.
This creates a grounding baseline that later generation must beat without losing
citation validity, abstention, or security.


> **Canonical learner route · module RAG-04 of 63**
>
> Required prerequisites: **RAG-03** · Previous: **RAG-03** ·
> Next after mastery: **RAG-05** · Expected first-pass workload:
> **5–8 hours**
>
> **Core path:** intuition, objectives, foundations, runnable implementation,
> failure modes, and assessed exercises. **Extension path:** history, production,
> tradeoffs, and interview material may be revisited after the core pass.
> Do not continue merely because every cell ran. Continue when you can complete
> the independent exercise and teach-back without notes. The canonical route in
> `docs/CURRICULUM_PATH.json` is authoritative when section-local file order and
> prerequisite order differ.


## 1 · Learning Objectives

Create gold answers and acceptable terms, generate extractive cited answers,
measure answer correctness/support/citation validity/abstention, distinguish
component failures, and block stale, injected, and unauthorized evidence.


<details>
<summary><strong>Mathematics notation support — open when a formula feels dense</strong></summary>

- $x_i$: item or component number $i$; a subscript is a label, not multiplication.
- $n$: number of examples; $d$: number of features or dimensions.
- $\mathbf x$: vector; $X$: matrix; $X^\top$: transpose (rows and columns swap).
- $\hat y$: an estimate or prediction; a hat marks an estimated quantity.
- $\sum$: add repeated terms; $\prod$: multiply repeated terms.
- $\lVert\mathbf x\rVert$: vector length; $|x|$: distance of one number from zero.
- $\frac{\partial f}{\partial x}$: slope of $f$ as $x$ changes while other inputs
  stay fixed; $\nabla f$: vector containing all parameter slopes.
- $P(A\mid B)$: probability of $A$ after restricting attention to cases where
  $B$ is true.
- $\log$ reverses an exponential and turns products into sums.

Read a formula one operator at a time, write object shapes beside vectors and
matrices, and substitute a tiny numeric example. Review PRE-01 through PRE-04 for
worked explanations of these symbols.
</details>


## Student Lesson Companion · RAG-04 · Grounded Answer and Citation Evaluation

Use this companion during the **core pass**. Write short answers before reading
the extension material; then correct them in a different colour after the lesson.

### Practical problem before history

1. What concrete task or decision are we trying to improve?
2. Why is it difficult with the data, time, labels, or compute available?
3. What is the simplest previous baseline?
4. Where does that baseline fail?
5. What capability must the new concept add?

Section 2 must supply enough evidence to answer these questions. Historical detail
is extension material unless it explains the problem or design constraint.

### Concept, analogy, and analogy limit

After Section 3, explain the concept in one sentence without unexplained jargon.
Map it to one daily-life analogy **or** one concrete visual example. Then state
one place where the mapping breaks; an analogy is a bridge, not a proof.

### Use / avoid / alternatives

Complete this decision table from Sections 7, 9, and 11:

| Decision | Required answer |
|---|---|
| Use it when | Three realistic conditions where its assumptions and benefits fit |
| Avoid it when | Two conditions where it is misleading, unsafe, or wasteful |
| Prefer instead | At least one simpler baseline and one alternative for a failed assumption |
| Evidence | Metric, diagnostic, or constraint that supports the choice |

### Code walkthrough and expected-result contract

For the first implementation, annotate: inputs → shapes/units → initialization →
central computation → intermediate output → final output → verification. Before
execution, record the expected value, range, or shape and what it would mean.

Distinguish these outcomes:

| Outcome | Interpretation | Next action |
|---|---|---|
| Exception, non-finite value, impossible shape | Code or data-contract failure | Inspect the first violated boundary |
| Output has valid type/shape but weak metric | Experiment ran; method may be poor | Diagnose data, assumptions, and baseline comparison |
| Strong metric on training data only | Insufficient evidence | Evaluate with the declared validation design |
| Expected output on untouched data | Supports one scoped claim | Record limitations; do not generalize beyond evidence |

### Debugging table

| Symptom | Likely cause | Inspect | Scoped fix |
|---|---|---|---|
| Shape/type error | Interface mismatch | Shapes, dtypes, feature names | Repair the boundary, not downstream symptoms |
| NaN/Inf or divergence | Invalid input or unstable update | Raw values, loss, gradients, learning rate | Clean/validate input or change one optimization control |
| Implausibly strong result | Leakage or invalid split | Fit boundaries, timestamps, duplicate entities | Rebuild the evaluation path before tuning |
| Different repeated result | Uncontrolled state | Seeds, data order, train/eval mode, versions | Record and control randomness intentionally |
| Plausible output but poor result | Wrong assumptions or representation | Baseline, slices, residuals/errors | Prefer a justified alternative; do not debug valid code as broken |


## 2 · Historical Motivation

Retrieval recall answers whether useful evidence appeared somewhere in the ranked
context. It does not prove the answer selected the right passage, interpreted it
correctly, cited it honestly, or refused an unsupported question. Treating one
final “accuracy” number as RAG evaluation hides the component that failed.


## 3 · Intuition and Visual Understanding

```text
query → ranked evidence → answer selection → answer → citations
             ↓                 ↓              ↓         ↓
         recall@k        correctness      support    validity
                unanswerable → abstention
```

A librarian can bring the right shelf but hand you the wrong book. Retrieval and
answering are related stages, not the same capability.


## 4 · Mathematical Foundations

Component success can be expressed as indicators. Citation validity is 1 when
every cited evidence ID is among retrieved IDs. Extractive support is 1 when the
answer is contained in cited evidence. Abstention accuracy is the fraction of
questions whose answer/refusal decision matches answerability.

For ten cases with nine correct answer/refusal decisions, abstention accuracy is
$9/10=0.9$. This does not measure whether the nine non-refused answers are correct.


## 5 · Manual Implementation from Scratch

Run the deterministic baseline and inspect the difference between high retrieval
recall and lower answer correctness.


In [ ]:
import sys
from pathlib import Path
ROOT = next(candidate for candidate in (Path.cwd(), *Path.cwd().parents)
            if (candidate / "projects/rag_foundations/src").exists())
sys.path.insert(0, str(ROOT / "projects/rag_foundations/src"))
from rag_foundations.grounded import evaluate_answers, evaluate_security

report = evaluate_answers(ROOT / "projects/rag_foundations/data")
print("metrics", report["metrics"])
print("failures", report["failure_counts"])


## 6 · Visualization

A component chart prevents a strong support score from hiding weak answer
selection. Perfect support is expected for extractive text and is not evidence of
correctness.


In [ ]:
import matplotlib.pyplot as plt
metric_names = [name for name in report["metrics"] if name != "mean_latency_ms"]
values = [report["metrics"][name] for name in metric_names]
plt.bar(metric_names, values); plt.ylim(0, 1.05); plt.xticks(rotation=45, ha="right")
plt.title("Component metrics reveal different failure modes")
plt.tight_layout(); plt.show()


## 7 · Failure Modes and Common Mistakes

- Calling retrieved evidence an answer.
- Scoring support without validating the citation ID.
- Counting an extractive quotation as correct merely because it is grounded.
- Letting untrusted document instructions control the system.
- Retrieving stale or unauthorized evidence before filtering.
- Tuning abstention on the final test cases.
- Reporting averages without per-query component traces.


## 8 · Library Implementation

`grounded.py` keeps answer text, evidence IDs, retrieved IDs, metrics, latency,
and outcome taxonomy in one trace. Later LLM generation must use the same schema
so its gains and regressions are comparable.


## 9 · Realistic Case Study

The curriculum assistant retrieves a relevant section for most questions, yet an
extractive top-one answer is correct on fewer cases. This shows that ranking the
correct section somewhere in top-k is not the same as choosing it for the answer.


## 10 · Production and Learning Considerations

Apply authorization, freshness, and unsafe-content controls before evidence
reaches answer generation. Log refusals without logging secrets. Version gold
answers separately from prompts and never rewrite labels to flatter a system.


## 11 · Tradeoff Analysis

Extractive answers maximize support and auditability but may be verbose or select
the wrong passage. Generative answers can synthesize multiple sources but add
interpretation, citation, and hallucination failures. Abstention improves safety
but can reduce useful coverage.


## 12 · Readiness and Interview Preparation

Given one trace, identify the first failed component. Recommend a retrieval,
selection, generation, citation, or policy fix—never “improve the RAG system” as
an undifferentiated response.


## 13 · Teach-Back

Explain retrieval recall, correctness, support, citation validity, and abstention
with one counterexample for each. Explain why prompt injection in retrieved text
is data to inspect, not an instruction to follow.


## 14 · Exercises, Self-Check, and Solutions

**Worked:** classify one trace into retrieval, answer, attribution, grounding, or
abstention failure. **Guided:** inspect five failed rows and identify the first
broken component. **Independent:** add an acceptable answer variant without
changing evidence labels. **Challenge:** add a deterministic multi-chunk answer
selector and prove whether correctness improves without lowering support/security.

<details><summary><strong>Solution and scoring rubric</strong></summary>
Full credit uses the first failed component, preserves gold labels, validates IDs,
and reports all component metrics. Award 3 points for taxonomy, 3 for trace-based
diagnosis, 2 for security, and 2 for controlled ablation. Common mistakes: using
gold evidence inside the runtime selector and treating exact extraction as correct
by definition. **Readiness threshold: 8/10.**
</details>


## Lesson Close · Summary, Student Check, and Memory Aid

### Five short student checks

1. What practical problem does **RAG-04 · Grounded Answer and Citation Evaluation** solve?
2. What is its central mechanism in simple language?
3. Which assumption or limitation is easiest to forget?
4. What output or diagnostic tells you it worked as intended?
5. When would you choose a simpler or related alternative?

### Plain-language summary

Complete four sentences without notes: **The problem is… The concept works by…
I would use it when… I would avoid it when…** Compare your answer with the
objectives, failure modes, tradeoff analysis, and teach-back section.

### One-sentence memory aid

**Remember RAG-04 · Grounded Answer and Citation Evaluation: start from the problem, trace the mechanism, verify the
evidence, and use it only when its assumptions fit.** Replace this general aid
with your own topic-specific sentence of no more than 20 words.

The lesson is complete only after the Required Core Mastery Gate, not after the
final code cell runs.


## Required Core Mastery Gate · RAG-04

Complete this gate before treating the module as finished. The longer exercises
in Section 14 are extension practice unless the module says otherwise.

**Worked example:** rerun the smallest worked example in this notebook. Annotate
every input, output, shape or unit, and the claim the result supports.

**Guided practice (20–30 min):** change one input in that example. Before running
it, predict the direction of change and explain why. Use the nearest preceding
formula or algorithm step as a hint. **Self-check:** compare prediction with the
result and explain any mismatch rather than editing the prediction afterward.

**Independent practice (30–45 min):** on fresh tiny data, reproduce the module's
central operation without copying the completed cell. State assumptions, expected
output, and one invalid input. **Self-check:** verify with an assertion, a manual
calculation, or a trusted library implementation.

**Challenge (30–60 min):** create one failure described in Section 7, detect it
using evidence, and repair it without changing unrelated variables.

**Solution and scoring rubric:** 2 points for a correct prediction, 2 for a
runnable independent implementation, 2 for an objective self-check, 2 for failure
diagnosis, and 2 for teach-back without notes. Common mistakes: copying before
attempting, using later-module concepts as unexplained shortcuts, evaluating on
training data, and continuing because cells ran. **Readiness threshold: 8/10**,
including both independent implementation and teach-back points. If below 8,
revisit the named prerequisite in the route card and retry with different data.
